# Week 8 Capstone — Meeting-to-Action Assistant

This notebook is the **self-contained submission artifact** for the COSC 650 capstone. It takes an existing meeting transcript and produces a structured operational brief containing:

- executive summary
- decisions
- action items and owners
- blockers
- follow-up questions

The goal is integration and measurement, not novelty. The system deliberately reuses earlier course components:

| Reused technique | Prior week | Capstone role |
|---|---:|---|
| Structured prompting/output | Week 3 | Stable operational-brief JSON contract |
| Deterministic validation | Week 4 | Validate required fields and fail visibly |
| RAG retrieval with MiniLM + FAISS | Week 5 | Retrieve task-relevant operational guidance |
| Cross-encoder reranking | Week 6 | Rerank retrieved guidance/examples before generation |
| Dynamic few-shot selection | Week 7 | Retrieve similar demonstrations for configuration C |

**Architecture**

```text
meeting transcript
      ↓
MiniLM + FAISS retrieval
      ↓
cross-encoder reranking
      ↓
optional dynamic few-shot examples
      ↓
Gemini Flash structured extraction
      ↓
deterministic validation
      ↓
metrics + LLM-as-judge + failure trace
```

The notebook also compares Gemini with a **small local Qwen model** on a narrower five-label classification task. That comparison measures whether a small CPU model is good enough for a limited role; it is not forced into the final architecture.

Out of scope: audio transcription, Teams/Zoom integration, automatic email/task execution, fine-tuning, deployment, web UI, or multi-agent orchestration.

In [ ]:
# Colab setup. Uncomment once in a fresh runtime if needed.
# %pip install -q sentence-transformers faiss-cpu openai pandas numpy tabulate transformers accelerate

import os, re, json, time, math, hashlib, random
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from IPython.display import display, Markdown

try:
    from google.colab import userdata
    if not os.getenv("GEMINI_API_KEY"):
        key = userdata.get("GEMINI_API_KEY")
        if key:
            os.environ["GEMINI_API_KEY"] = key
except Exception:
    pass

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")
GEMINI_JUDGE_MODEL = os.getenv("GEMINI_JUDGE_MODEL", GEMINI_MODEL)

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Small local CPU model used only for the narrow comparison.
LOCAL_QWEN_MODEL = os.getenv("LOCAL_QWEN_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")

# Full uncached cap:
# 30 synthesis calls = 10 held-out cases × 3 ablations
# 10 judge calls = judge the selected configuration on 10 held-out cases
# 1 Gemini batch-classification call for the Qwen comparison
MAX_SYNTHESIS_CALLS = 30
MAX_JUDGE_CALLS = 10
MAX_CLASSIFIER_CALLS = 1

# The cache lives inside the notebook runtime.
# After a measured run, copy CACHE_EXPORT into PRELOADED_CACHE_JSON before committing
# if you want a clean rerun to avoid repeating completed Gemini calls.
PRELOADED_CACHE_JSON = ""
CACHE = json.loads(PRELOADED_CACHE_JSON) if PRELOADED_CACHE_JSON.strip() else {
    "synthesis": {},
    "judge": {},
    "classifier": {}
}

LIVE_SYNTHESIS = 0
LIVE_JUDGE = 0
LIVE_CLASSIFIER = 0

def cache_key(payload):
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")
    ).hexdigest()

def api_client():
    from openai import OpenAI
    if not os.getenv("GEMINI_API_KEY"):
        raise RuntimeError("GEMINI_API_KEY is not set.")
    return OpenAI(
        api_key=os.environ["GEMINI_API_KEY"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )

def gemini_call_with_retry(model, messages, temperature=0, max_attempts=6):
    """Retry transient Gemini/API failures without hiding permanent errors."""
    delays = [2, 4, 8, 16, 32]
    for attempt in range(max_attempts):
        try:
            return api_client().chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature
            )
        except Exception as e:
            status = getattr(e, "status_code", None)
            transient = status in {429, 500, 502, 503, 504}
            if not transient or attempt == max_attempts - 1:
                raise
            delay = delays[min(attempt, len(delays)-1)] + random.uniform(0, 1)
            print(f"Gemini temporary error {status}; retrying in {delay:.1f}s...")
            time.sleep(delay)

def parse_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    return json.loads(text)

print("Gemini synthesis:", GEMINI_MODEL)
print("Gemini judge:", GEMINI_JUDGE_MODEL)
print("Local Qwen:", LOCAL_QWEN_MODEL)

## Held-out evaluation set

The 10 cases below are frozen before model execution. They are not used as dynamic few-shot demonstrations.

The cases intentionally include ambiguous language: suggestions that should not become decisions, cross-speaker ownership, unresolved questions, blockers mixed with status, and meetings with no confirmed decision.

In [ ]:
EVAL_CASES = [
    {
        "id":"E01",
        "transcript":"""Brittany: We need the renewal demo ready by Friday.
Michael: I can finish the validation worksheet Thursday.
Brittany: Good. Michael owns the worksheet, and we'll use the current pricing template for the demo.
Michael: The OpenAir export is still missing two customer IDs.
Brittany: Ask Finance whether those IDs can be corrected before Thursday.""",
        "gold":{
            "decisions":["Use the current pricing template for the renewal demo."],
            "actions":[
                {"description":"Finish the validation worksheet by Thursday.","owner":"Michael"},
                {"description":"Ask Finance whether the missing customer IDs can be corrected before Thursday.","owner":"Michael"}
            ],
            "blockers":["The OpenAir export is missing two customer IDs."],
            "questions":["Can Finance correct the missing customer IDs before Thursday?"]
        }
    },
    {
        "id":"E02",
        "transcript":"""Alex: Maybe we should move the launch to Monday.
Priya: I would rather keep Friday unless QA finds something serious.
Alex: Agreed. No schedule change right now.
Priya: I'll send the final QA result by 3 PM.""",
        "gold":{
            "decisions":["Keep the Friday launch schedule unless QA finds a serious issue."],
            "actions":[{"description":"Send the final QA result by 3 PM.","owner":"Priya"}],
            "blockers":[],
            "questions":[]
        }
    },
    {
        "id":"E03",
        "transcript":"""Sam: The customer asked whether SSO can be included in phase one.
Jordan: We don't know the engineering effort yet.
Sam: Could we promise it conditionally?
Jordan: No. Let's get an estimate first. Jordan will ask Engineering for the effort by Wednesday.""",
        "gold":{
            "decisions":["Do not promise SSO for phase one until an engineering estimate is available."],
            "actions":[{"description":"Ask Engineering for the SSO effort estimate by Wednesday.","owner":"Jordan"}],
            "blockers":["The engineering effort for SSO is unknown."],
            "questions":["What is the engineering effort for adding SSO in phase one?"]
        }
    },
    {
        "id":"E04",
        "transcript":"""Nina: I think a weekly dashboard could help.
Omar: Possibly. Let's look at examples next week.
Nina: There is no decision on dashboard cadence today.
Omar: I'll collect three dashboard examples before our next meeting.""",
        "gold":{
            "decisions":[],
            "actions":[{"description":"Collect three dashboard examples before the next meeting.","owner":"Omar"}],
            "blockers":[],
            "questions":[]
        }
    },
    {
        "id":"E05",
        "transcript":"""Lee: The import failed because three rows have invalid dates.
Casey: Can we fix them manually?
Lee: Yes. I'll correct the dates and rerun the import today.
Casey: Approved. Keep the current import mapping; don't redesign it for this release.""",
        "gold":{
            "decisions":["Keep the current import mapping for this release."],
            "actions":[{"description":"Correct the invalid dates and rerun the import today.","owner":"Lee"}],
            "blockers":["Three rows have invalid dates and caused the import to fail."],
            "questions":[]
        }
    },
    {
        "id":"E06",
        "transcript":"""Maya: We have two vendors that could supply the API data.
Chris: Vendor B is cheaper, but legal hasn't reviewed its terms.
Maya: Let's not choose a vendor yet.
Chris: I'll send both agreements to Legal.
Maya: Which agreement can Legal clear this week?""",
        "gold":{
            "decisions":["Do not choose an API data vendor yet."],
            "actions":[{"description":"Send both vendor agreements to Legal.","owner":"Chris"}],
            "blockers":["Legal has not reviewed Vendor B's terms."],
            "questions":["Which vendor agreement can Legal clear this week?"]
        }
    },
    {
        "id":"E07",
        "transcript":"""Dana: The pilot users liked the shorter form.
Eli: Should we remove the optional notes field too?
Dana: Not yet. We did agree to keep the shorter form for the pilot.
Eli: I'll measure completion time for another week.""",
        "gold":{
            "decisions":["Keep the shorter form for the pilot."],
            "actions":[{"description":"Measure form completion time for another week.","owner":"Eli"}],
            "blockers":[],
            "questions":["Should the optional notes field be removed?"]
        }
    },
    {
        "id":"E08",
        "transcript":"""Rosa: The report numbers from Analytics and Search Console don't match.
Ian: The date ranges are different.
Rosa: Use September 1 through September 30 for both sources.
Ian: Got it. I'll rerun both reports with that date range and send the numbers tomorrow.""",
        "gold":{
            "decisions":["Use September 1 through September 30 as the date range for both sources."],
            "actions":[{"description":"Rerun both reports with the September 1 through September 30 date range and send the numbers tomorrow.","owner":"Ian"}],
            "blockers":["Analytics and Search Console report numbers do not match because the date ranges differ."],
            "questions":[]
        }
    },
    {
        "id":"E09",
        "transcript":"""Taylor: We could automate the email after generation.
Morgan: I don't want automatic sending in version one.
Taylor: Okay, generated drafts will require human review.
Morgan: Taylor, document that approval gate in the design.""",
        "gold":{
            "decisions":["Version one will not automatically send generated email; generated drafts require human review."],
            "actions":[{"description":"Document the human approval gate in the design.","owner":"Taylor"}],
            "blockers":[],
            "questions":[]
        }
    },
    {
        "id":"E10",
        "transcript":"""Avery: The customer list has duplicate organizations.
Quinn: That explains the mismatched renewal totals.
Avery: We need a matching key before proposal generation.
Quinn: I'll test domain plus customer ID as the matching key.
Avery: Is customer ID populated for every record?""",
        "gold":{
            "decisions":["A matching key is required before proposal generation."],
            "actions":[{"description":"Test domain plus customer ID as the matching key.","owner":"Quinn"}],
            "blockers":["Duplicate organizations are causing mismatched renewal totals."],
            "questions":["Is customer ID populated for every record?"]
        }
    }
]

assert 10 <= len(EVAL_CASES) <= 20
assert len({x["id"] for x in EVAL_CASES}) == len(EVAL_CASES)
print("Held-out cases:", len(EVAL_CASES))

## Reused retrieval and few-shot components

The RAG corpus contains task rules rather than external factual knowledge. This makes the ablation measurable: retrieval either helps the model apply operational definitions or it does not.

The few-shot pool is separate from the held-out set.

In [ ]:
RULE_DOCS = [
    "A decision requires commitment, approval, agreement, selection, rejection, or an explicit statement that a course of action will be followed.",
    "A suggestion, possibility, brainstorm, or hypothetical such as maybe, could, might, consider, or what if is not a confirmed decision unless a later turn explicitly commits to it.",
    "An action item is future work assigned or accepted by a person or team. Capture the work and the owner; do not invent an owner when none is supported.",
    "Owner resolution may require cross-speaker context. Phrases such as 'I'll do it' refer to the current speaker even when the task was introduced earlier.",
    "A blocker is a concrete condition preventing, delaying, or risking progress. General background is not automatically a blocker.",
    "A follow-up question is an unresolved question that still needs an answer after the meeting.",
    "Every extracted decision, action, blocker, and follow-up question must be grounded in transcript evidence."
]

FEW_SHOT_EXAMPLES = [
    ("A: Maybe delay launch. B: No, keep Friday. A: Okay.",
     "Decision: keep the Friday launch; the delay was only a suggestion."),
    ("A: Could we use vendor B? B: Let's compare legal terms first.",
     "No vendor decision was made."),
    ("A: The export is missing IDs. B: I'll ask Finance to correct them.",
     "Action item: B will ask Finance to correct the missing IDs."),
    ("A: Legal has not reviewed the vendor terms.",
     "Blocker: legal review of the vendor terms is outstanding."),
    ("A: Which agreement can Legal clear this week?",
     "Unresolved follow-up question: which agreement can Legal clear this week?"),
    ("A: We agreed to keep human review before email is sent.",
     "Decision: generated email requires human review before sending."),
    ("A: Maybe remove the notes field. B: Not yet.",
     "Removing the notes field was discussed but not approved."),
    ("A: Use September 1 through September 30 for both reports. B: Got it.",
     "Decision: both reports will use September 1 through September 30.")
]

embedder = SentenceTransformer(EMBED_MODEL)
reranker = CrossEncoder(RERANK_MODEL)

rule_vectors = embedder.encode(RULE_DOCS, normalize_embeddings=True).astype("float32")
rule_index = faiss.IndexFlatIP(rule_vectors.shape[1])
rule_index.add(rule_vectors)

shot_vectors = embedder.encode([x[0] for x in FEW_SHOT_EXAMPLES], normalize_embeddings=True).astype("float32")
shot_index = faiss.IndexFlatIP(shot_vectors.shape[1])
shot_index.add(shot_vectors)

def retrieve_and_rerank(query, docs, index, text_fn=lambda x: x, candidate_k=6, top_k=4):
    qv = embedder.encode([query], normalize_embeddings=True).astype("float32")
    dense_scores, ids = index.search(qv, min(candidate_k, len(docs)))
    candidates = [docs[int(i)] for i in ids[0] if int(i) >= 0]
    scores = reranker.predict([(query, text_fn(x)) for x in candidates])
    ranked = sorted(zip(scores, candidates), key=lambda z: float(z[0]), reverse=True)
    return [x for _, x in ranked[:top_k]]

def retrieve_rules(transcript):
    return retrieve_and_rerank(transcript, RULE_DOCS, rule_index, top_k=4)

def retrieve_examples(transcript):
    return retrieve_and_rerank(
        transcript, FEW_SHOT_EXAMPLES, shot_index,
        text_fn=lambda x: x[0], top_k=3
    )

display(pd.DataFrame({"retrieved_rule": retrieve_rules(EVAL_CASES[1]["transcript"])}))

## Structured extraction and deterministic validation

Gemini handles ambiguous interpretation and synthesis. Python handles the output contract, matching, metrics, caching, and failure analysis.

In [ ]:
BASE_PROMPT = """Convert the meeting transcript into a structured operational brief.

Rules:
- Report only confirmed decisions, not suggestions or possibilities.
- Capture future work as action items and identify the supported owner.
- Do not invent an owner. Use "Unknown" when the transcript does not support one.
- A blocker must materially prevent, delay, or risk progress.
- A follow-up question must still be unresolved at the end of the transcript.
- Every extracted item must contain concise transcript evidence.
- Return JSON only.

Exact JSON shape:
{
  "executive_summary": "string",
  "decisions": [{"decision": "string", "evidence": "string"}],
  "action_items": [{"description": "string", "owner": "string", "evidence": "string"}],
  "blockers": [{"blocker": "string", "evidence": "string"}],
  "follow_up_questions": [{"question": "string", "evidence": "string"}]
}
"""

def validate_brief(obj):
    required = {
        "executive_summary": str,
        "decisions": list,
        "action_items": list,
        "blockers": list,
        "follow_up_questions": list
    }
    if not isinstance(obj, dict):
        return False, ["top level is not an object"]

    errors = []
    for key, typ in required.items():
        if key not in obj:
            errors.append(f"missing field: {key}")
        elif not isinstance(obj[key], typ):
            errors.append(f"{key} must be {typ.__name__}")

    for i, x in enumerate(obj.get("decisions", []) if isinstance(obj.get("decisions"), list) else []):
        for k in ("decision", "evidence"):
            if not isinstance(x, dict) or not isinstance(x.get(k), str) or not x.get(k, "").strip():
                errors.append(f"decisions[{i}].{k} invalid")

    for i, x in enumerate(obj.get("action_items", []) if isinstance(obj.get("action_items"), list) else []):
        for k in ("description", "owner", "evidence"):
            if not isinstance(x, dict) or not isinstance(x.get(k), str) or not x.get(k, "").strip():
                errors.append(f"action_items[{i}].{k} invalid")

    for field, text_key in (("blockers", "blocker"), ("follow_up_questions", "question")):
        for i, x in enumerate(obj.get(field, []) if isinstance(obj.get(field), list) else []):
            for k in (text_key, "evidence"):
                if not isinstance(x, dict) or not isinstance(x.get(k), str) or not x.get(k, "").strip():
                    errors.append(f"{field}[{i}].{k} invalid")

    return len(errors) == 0, errors

def extract_brief(transcript, config):
    global LIVE_SYNTHESIS

    ck = cache_key({
        "kind":"synthesis",
        "model":GEMINI_MODEL,
        "config":config,
        "transcript":transcript
    })
    if ck in CACHE["synthesis"]:
        out = dict(CACHE["synthesis"][ck])
        out["cache_hit"] = True
        return out

    if LIVE_SYNTHESIS >= MAX_SYNTHESIS_CALLS:
        raise RuntimeError("Synthesis live-call cap reached.")

    parts = [BASE_PROMPT]
    retrieval_log = {"rules": [], "examples": []}

    if config in ("B", "C"):
        retrieval_log["rules"] = retrieve_rules(transcript)
        parts.append("Retrieved operational guidance:\n" + "\n".join(
            f"- {x}" for x in retrieval_log["rules"]
        ))

    if config == "C":
        retrieval_log["examples"] = retrieve_examples(transcript)
        parts.append("Retrieved examples:\n" + "\n".join(
            f"- Input: {a}\n  Interpretation: {b}"
            for a, b in retrieval_log["examples"]
        ))

    parts.append("Transcript:\n" + transcript)

    t0 = time.perf_counter()
    response = gemini_call_with_retry(
        GEMINI_MODEL,
        [{"role":"user", "content":"\n\n".join(parts)}],
        temperature=0
    )
    latency = time.perf_counter() - t0
    raw = response.choices[0].message.content

    try:
        brief = parse_json(raw)
        parse_error = None
    except Exception as e:
        brief = {}
        parse_error = f"{type(e).__name__}: {e}"

    valid, validation_errors = validate_brief(brief)
    out = {
        "brief": brief,
        "raw": raw,
        "valid": valid,
        "validation_errors": validation_errors,
        "parse_error": parse_error,
        "latency_s": latency,
        "retrieval": retrieval_log
    }

    CACHE["synthesis"][ck] = out
    LIVE_SYNTHESIS += 1

    out = dict(out)
    out["cache_hit"] = False
    return out

## Deterministic metrics

The A/B/C ablation compares:

1. **A — transcript only**
2. **B — transcript + retrieved/reranked operational guidance**
3. **C — transcript + retrieved/reranked guidance + dynamic few-shot examples**

Metrics include decision and action-item precision/recall/F1, owner accuracy, blocker F1, follow-up-question F1, structured-output validity, and latency.

In [ ]:
STOP = {"the","a","an","to","of","and","or","for","in","on","by","with","is","are","be","as","at","this","that"}

def tokens(s):
    return [x for x in re.findall(r"[a-z0-9]+", str(s).lower()) if x not in STOP]

def token_jaccard(a, b):
    A, B = set(tokens(a)), set(tokens(b))
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def best_match_pairs(pred, gold, threshold=0.42):
    pairs = []
    unused = set(range(len(gold)))
    for pi, p in enumerate(pred):
        scored = [(token_jaccard(p, gold[gi]), gi) for gi in unused]
        if scored:
            score, gi = max(scored)
            if score >= threshold:
                pairs.append((pi, gi, score))
                unused.remove(gi)
    return pairs

def prf(pred, gold, threshold=0.42):
    pairs = best_match_pairs(pred, gold, threshold)
    tp = len(pairs)
    precision = tp / len(pred) if pred else (1.0 if not gold else 0.0)
    recall = tp / len(gold) if gold else (1.0 if not pred else 0.0)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1, pairs

def flatten_brief(brief):
    return {
        "decisions":[x.get("decision","") for x in brief.get("decisions",[]) if isinstance(x,dict)],
        "actions":[x.get("description","") for x in brief.get("action_items",[]) if isinstance(x,dict)],
        "owners":[x.get("owner","") for x in brief.get("action_items",[]) if isinstance(x,dict)],
        "blockers":[x.get("blocker","") for x in brief.get("blockers",[]) if isinstance(x,dict)],
        "questions":[x.get("question","") for x in brief.get("follow_up_questions",[]) if isinstance(x,dict)]
    }

def score_case(case, result):
    gold = case["gold"]
    pred = flatten_brief(result["brief"]) if result.get("valid") else {
        "decisions":[], "actions":[], "owners":[], "blockers":[], "questions":[]
    }

    dp, dr, df, _ = prf(pred["decisions"], gold["decisions"])
    ap, ar, af, action_pairs = prf(
        pred["actions"],
        [x["description"] for x in gold["actions"]]
    )
    _, _, bf, _ = prf(pred["blockers"], gold["blockers"])
    _, _, qf, _ = prf(pred["questions"], gold["questions"])

    owner_hits = []
    for pi, gi, _ in action_pairs:
        pred_owner = pred["owners"][pi].strip().lower() if pi < len(pred["owners"]) else ""
        gold_owner = gold["actions"][gi]["owner"].strip().lower()
        owner_hits.append(pred_owner == gold_owner)
    owner_acc = (
        sum(owner_hits) / len(owner_hits)
        if owner_hits
        else (1.0 if not gold["actions"] else 0.0)
    )

    return {
        "case_id":case["id"],
        "decision_precision":dp,
        "decision_recall":dr,
        "decision_f1":df,
        "action_precision":ap,
        "action_recall":ar,
        "action_f1":af,
        "owner_accuracy":owner_acc,
        "blocker_f1":bf,
        "question_f1":qf,
        "valid":float(result.get("valid",False)),
        "latency_s":result.get("latency_s",math.nan)
    }

RESULTS = {}
rows = []

if not os.getenv("GEMINI_API_KEY"):
    print("API-BLOCKED: add GEMINI_API_KEY, then rerun this cell.")
else:
    for cfg in ("A","B","C"):
        RESULTS[cfg] = {}
        for case in EVAL_CASES:
            try:
                result = extract_brief(case["transcript"], cfg)
            except Exception as e:
                print(f"{cfg}/{case['id']} failed after retries: {e}")
                result = {
                    "brief":{},
                    "raw":"",
                    "valid":False,
                    "validation_errors":[str(e)],
                    "parse_error":str(e),
                    "latency_s":math.nan,
                    "retrieval":{"rules":[],"examples":[]},
                    "cache_hit":False
                }
            RESULTS[cfg][case["id"]] = result
            s = score_case(case, result)
            s["config"] = cfg
            s["cache_hit"] = bool(result.get("cache_hit", False))
            rows.append(s)

    METRICS = pd.DataFrame(rows)
    metric_cols = [
        "decision_precision","decision_recall","decision_f1",
        "action_precision","action_recall","action_f1",
        "owner_accuracy","blocker_f1","question_f1",
        "valid","latency_s"
    ]
    SUMMARY = METRICS.groupby("config")[metric_cols].mean().round(3)
    display(METRICS.round(3))
    display(SUMMARY)

    # Objective selection rule for the configuration sent to the judge.
    SUMMARY["extraction_score"] = SUMMARY[
        ["decision_f1","action_f1","owner_accuracy","blocker_f1","question_f1","valid"]
    ].mean(axis=1)
    SELECTED_CONFIG = SUMMARY["extraction_score"].idxmax()
    print("Selected configuration for judge pass:", SELECTED_CONFIG)

## LLM-as-judge pass

The rubric requires an LLM-as-judge pass over 10–20 held-out cases on named dimensions.

To control free-tier calls, the judge evaluates the **deterministically selected configuration** on all 10 held-out cases rather than judging all three ablations. The dimensions are:

1. **correctness**
2. **grounding**
3. **operational usefulness**

In [ ]:
JUDGE_PROMPT = """Score the candidate meeting brief from 1 to 5 on exactly three dimensions.

correctness:
5 = operational content is essentially correct
1 = major decisions/actions/owners are wrong or missing

grounding:
5 = all material claims are supported by the transcript
1 = substantial unsupported or invented content

operational_usefulness:
5 = concise and immediately actionable
1 = confusing, incomplete, or operationally misleading

Return JSON only:
{"correctness": <1-5>, "grounding": <1-5>, "operational_usefulness": <1-5>, "reason": "brief reason"}
"""

def judge_case(case, candidate):
    global LIVE_JUDGE

    payload = {
        "transcript":case["transcript"],
        "gold_reference":case["gold"],
        "candidate":candidate
    }

    ck = cache_key({
        "kind":"judge",
        "model":GEMINI_JUDGE_MODEL,
        "payload":payload
    })
    if ck in CACHE["judge"]:
        out = dict(CACHE["judge"][ck])
        out["cache_hit"] = True
        return out

    if LIVE_JUDGE >= MAX_JUDGE_CALLS:
        raise RuntimeError("Judge live-call cap reached.")

    response = gemini_call_with_retry(
        GEMINI_JUDGE_MODEL,
        [{"role":"user","content":JUDGE_PROMPT + "\n\n" + json.dumps(payload, ensure_ascii=False)}],
        temperature=0
    )
    out = parse_json(response.choices[0].message.content)
    CACHE["judge"][ck] = out
    LIVE_JUDGE += 1

    out = dict(out)
    out["cache_hit"] = False
    return out

JUDGE_DF = pd.DataFrame()
JUDGE_SUMMARY = pd.DataFrame()

if RESULTS:
    judge_rows = []
    case_map = {x["id"]:x for x in EVAL_CASES}

    for case in EVAL_CASES:
        try:
            j = judge_case(case, RESULTS[SELECTED_CONFIG][case["id"]]["brief"])
            judge_rows.append({
                "case_id":case["id"],
                **j
            })
        except Exception as e:
            print(f"Judge/{case['id']} failed after retries: {e}")

    JUDGE_DF = pd.DataFrame(judge_rows)
    if not JUDGE_DF.empty:
        display(JUDGE_DF)
        JUDGE_SUMMARY = JUDGE_DF[
            ["correctness","grounding","operational_usefulness"]
        ].mean().to_frame("average").round(3)
        display(JUDGE_SUMMARY)

## Required failure mode and mitigation

The declared failure is a **suggestion or discussion point being incorrectly promoted to a confirmed decision**.

The trace uses the A/B/C comparison:

- C worse than B → dynamic few-shot context is implicated.
- B worse than A → retrieved/reranked context is implicated.
- Otherwise → the lowest-scoring case is attributed to extraction/generation rather than inventing a retrieval failure.

The mitigation is deliberately narrow: remove a predicted decision when its own evidence contains hedging/suggestion language without a commitment cue.

In [ ]:
FAILURE = None
MITIGATION_BEFORE = None
MITIGATION_AFTER = None

if RESULTS:
    scores = {
        cfg:{
            case["id"]:score_case(case, RESULTS[cfg][case["id"]])
            for case in EVAL_CASES
        }
        for cfg in ("A","B","C")
    }

    chosen = None
    component = None

    for case in EVAL_CASES:
        cid = case["id"]
        if scores["C"][cid]["decision_f1"] < scores["B"][cid]["decision_f1"]:
            chosen, component = cid, "dynamic few-shot context"
            break

    if chosen is None:
        for case in EVAL_CASES:
            cid = case["id"]
            if scores["B"][cid]["decision_f1"] < scores["A"][cid]["decision_f1"]:
                chosen, component = cid, "retrieved/reranked context"
                break

    if chosen is None:
        chosen = min(
            (case["id"] for case in EVAL_CASES),
            key=lambda cid: scores[SELECTED_CONFIG][cid]["decision_f1"]
        )
        component = "generation/extraction stage"

    FAILURE = {"case_id":chosen, "component":component}

    print("Failure case:", chosen)
    print("Responsible component:", component)

    case = next(x for x in EVAL_CASES if x["id"] == chosen)
    print("\nTranscript:\n", case["transcript"])
    print("\nGold:\n", json.dumps(case["gold"], indent=2))

    for cfg in ("A","B","C"):
        print(f"\n--- {cfg} ---")
        print(json.dumps(RESULTS[cfg][chosen]["brief"], indent=2))
        print("scores:", scores[cfg][chosen])
        if cfg in ("B","C"):
            print("retrieved rules:", RESULTS[cfg][chosen]["retrieval"].get("rules", []))
        if cfg == "C":
            print("retrieved examples:", RESULTS[cfg][chosen]["retrieval"].get("examples", []))

HEDGE_CUES = ["maybe","could","might","consider","possibly","what if","i think","would rather"]
COMMIT_CUES = ["agreed","approved","decided","we will","we'll","keep","use ","do not","don't","required","requires"]

def mitigate_decisions(brief):
    out = json.loads(json.dumps(brief))
    kept = []
    removed = []

    for item in out.get("decisions", []):
        evidence = item.get("evidence","").lower() if isinstance(item,dict) else str(item).lower()
        hedged = any(x in evidence for x in HEDGE_CUES)
        committed = any(x in evidence for x in COMMIT_CUES)
        if hedged and not committed:
            removed.append(item)
        else:
            kept.append(item)

    out["decisions"] = kept
    return out, removed

if RESULTS:
    before_rows, after_rows = [], []

    for case in EVAL_CASES:
        raw = RESULTS[SELECTED_CONFIG][case["id"]]
        before_rows.append(score_case(case, raw))

        mitigated, removed = mitigate_decisions(raw["brief"]) if raw.get("valid") else ({}, [])
        result2 = dict(raw)
        result2["brief"] = mitigated
        result2["valid"], result2["validation_errors"] = validate_brief(mitigated) if mitigated else (False, ["invalid"])
        after_rows.append(score_case(case, result2))

    before_df = pd.DataFrame(before_rows)
    after_df = pd.DataFrame(after_rows)

    MITIGATION_BEFORE = round(before_df["decision_f1"].mean(), 3)
    MITIGATION_AFTER = round(after_df["decision_f1"].mean(), 3)

    print("Mean decision F1 before mitigation:", MITIGATION_BEFORE)
    print("Mean decision F1 after mitigation:", MITIGATION_AFTER)

## Small local Qwen comparison

This comparison asks a narrower question:

> Can a small CPU-only Qwen model handle span classification well enough to take a limited local role, while Gemini remains responsible for full meeting interpretation and synthesis?

The shared classification labels are:

- `decision`
- `action_item`
- `blocker`
- `follow_up_question`
- `context`

Gemini classifies the complete 20-item set in one batched call. Qwen runs locally on CPU with no API cost.

In [ ]:
CLASSIFICATION_SET = [
    ("Maybe we should move the launch to Monday.", "context"),
    ("No schedule change right now.", "decision"),
    ("I'll send the final QA result by 3 PM.", "action_item"),
    ("The engineering effort for SSO is unknown.", "blocker"),
    ("What is the engineering effort for SSO?", "follow_up_question"),
    ("There is no decision on dashboard cadence today.", "context"),
    ("Keep the current import mapping for this release.", "decision"),
    ("I'll correct the dates and rerun the import today.", "action_item"),
    ("Legal has not reviewed Vendor B's terms.", "blocker"),
    ("Which agreement can Legal clear this week?", "follow_up_question"),
    ("We agreed to keep the shorter form for the pilot.", "decision"),
    ("I'll measure completion time for another week.", "action_item"),
    ("The report numbers do not match because the date ranges differ.", "blocker"),
    ("Use September 1 through September 30 for both sources.", "decision"),
    ("Could we automate the email after generation?", "context"),
    ("Generated drafts will require human review.", "decision"),
    ("I'll test domain plus customer ID as the matching key.", "action_item"),
    ("Is customer ID populated for every record?", "follow_up_question"),
    ("The owner field is wrong on two cases.", "blocker"),
    ("Do we need a separate coreference model?", "follow_up_question")
]

VALID_LABELS = {
    "decision","action_item","blocker","follow_up_question","context"
}

def gemini_classify_batch(items):
    global LIVE_CLASSIFIER

    payload = [text for text, _ in items]
    ck = cache_key({
        "kind":"classifier",
        "model":GEMINI_MODEL,
        "items":payload
    })

    if ck in CACHE["classifier"]:
        return dict(CACHE["classifier"][ck])

    if LIVE_CLASSIFIER >= MAX_CLASSIFIER_CALLS:
        raise RuntimeError("Classifier live-call cap reached.")

    prompt = """Classify each span into exactly one label:
decision, action_item, blocker, follow_up_question, context.

Return JSON only:
{"labels": ["label1", "label2", ...]}

Return exactly one label for each input span, in the same order.

Spans:
""" + "\n".join(f"{i+1}. {text}" for i, (text, _) in enumerate(items))

    t0 = time.perf_counter()
    response = gemini_call_with_retry(
        GEMINI_MODEL,
        [{"role":"user","content":prompt}],
        temperature=0
    )
    latency = time.perf_counter() - t0

    obj = parse_json(response.choices[0].message.content)
    labels = [str(x).strip().lower() for x in obj["labels"]]

    result = {"labels":labels, "latency_s":latency}
    CACHE["classifier"][ck] = result
    LIVE_CLASSIFIER += 1
    return result

GEMINI_CLASSIFIER_DF = pd.DataFrame()
QWEN_DF = pd.DataFrame()
MODEL_COMPARISON = pd.DataFrame()

if os.getenv("GEMINI_API_KEY"):
    try:
        g = gemini_classify_batch(CLASSIFICATION_SET)
        rows = []
        per_item_latency = g["latency_s"] / len(CLASSIFICATION_SET)
        for (text, gold), pred in zip(CLASSIFICATION_SET, g["labels"]):
            rows.append({
                "text":text,
                "gold":gold,
                "pred":pred,
                "correct":pred == gold,
                "latency_s":per_item_latency
            })
        GEMINI_CLASSIFIER_DF = pd.DataFrame(rows)
        display(GEMINI_CLASSIFIER_DF)
    except Exception as e:
        print("Gemini classification comparison failed:", e)

# Local Qwen CPU benchmark.
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    qwen_tokenizer = AutoTokenizer.from_pretrained(LOCAL_QWEN_MODEL)
    qwen_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_QWEN_MODEL,
        torch_dtype="auto",
        device_map=None
    )
    qwen_model.eval()

    def qwen_classify(text):
        messages = [
            {
                "role":"system",
                "content":"Classify the span into exactly one label: decision, action_item, blocker, follow_up_question, context. Return only the label."
            },
            {"role":"user","content":text}
        ]
        rendered = qwen_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = qwen_tokenizer(rendered, return_tensors="pt")
        t0 = time.perf_counter()
        with torch.no_grad():
            output = qwen_model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False
            )
        latency = time.perf_counter() - t0
        generated = output[0][inputs["input_ids"].shape[1]:]
        pred = qwen_tokenizer.decode(generated, skip_special_tokens=True).strip().lower()
        pred = re.sub(r"[^a-z_]", "", pred)
        return pred, latency

    qrows = []
    for text, gold in CLASSIFICATION_SET:
        pred, latency = qwen_classify(text)
        qrows.append({
            "text":text,
            "gold":gold,
            "pred":pred,
            "correct":pred == gold,
            "latency_s":latency
        })

    QWEN_DF = pd.DataFrame(qrows)
    display(QWEN_DF)

except Exception as e:
    print("Local Qwen comparison failed:", e)

comparison_rows = []

if not GEMINI_CLASSIFIER_DF.empty:
    comparison_rows.append({
        "model":"Gemini Flash",
        "task":"20-span classification",
        "accuracy":GEMINI_CLASSIFIER_DF["correct"].mean(),
        "mean_latency_s":GEMINI_CLASSIFIER_DF["latency_s"].mean(),
        "api_cost":"free-tier API"
    })

if not QWEN_DF.empty:
    comparison_rows.append({
        "model":LOCAL_QWEN_MODEL,
        "task":"20-span classification",
        "accuracy":QWEN_DF["correct"].mean(),
        "mean_latency_s":QWEN_DF["latency_s"].mean(),
        "api_cost":"$0 local inference"
    })

MODEL_COMPARISON = pd.DataFrame(comparison_rows)
if not MODEL_COMPARISON.empty:
    display(MODEL_COMPARISON.round(3))

## Research note

The research note is generated dynamically from the executed measurements. It answers the project questions rather than containing placeholders.

In [ ]:
def build_research_note():
    if not RESULTS or JUDGE_DF.empty or FAILURE is None:
        display(Markdown(
            "### Measured results\n\n"
            "The required evaluation has not completed yet. Run the A/B/C evaluation, "
            "judge pass, failure trace, and local-model comparison first."
        ))
        return None

    metric_cols = [
        "decision_precision","decision_recall","decision_f1",
        "action_precision","action_recall","action_f1",
        "owner_accuracy","blocker_f1","question_f1",
        "valid","latency_s"
    ]
    det = METRICS.groupby("config")[metric_cols].mean().round(3)

    # Retrieval effect: B vs A. Few-shot effect: C vs B.
    core_cols = ["decision_f1","action_f1","owner_accuracy","blocker_f1","question_f1","valid"]
    a_score = float(det.loc["A", core_cols].mean())
    b_score = float(det.loc["B", core_cols].mean())
    c_score = float(det.loc["C", core_cols].mean())

    retrieval_answer = (
        f"Yes. The aggregate extraction score increased from {a_score:.3f} to {b_score:.3f}."
        if b_score > a_score
        else (
            f"No measurable improvement. The aggregate extraction score changed from {a_score:.3f} to {b_score:.3f}."
        )
    )

    fewshot_answer = (
        f"Yes. Configuration C improved the aggregate extraction score from {b_score:.3f} to {c_score:.3f}."
        if c_score > b_score
        else (
            f"No. Configuration C changed the aggregate extraction score from {b_score:.3f} to {c_score:.3f}; the added few-shot context did not improve the measured result."
        )
    )

    mitigation_answer = (
        f"Yes. Mean decision F1 improved from {MITIGATION_BEFORE:.3f} to {MITIGATION_AFTER:.3f}."
        if MITIGATION_AFTER is not None and MITIGATION_BEFORE is not None and MITIGATION_AFTER > MITIGATION_BEFORE
        else (
            f"No improvement was measured. Mean decision F1 changed from {MITIGATION_BEFORE:.3f} to {MITIGATION_AFTER:.3f}."
            if MITIGATION_AFTER is not None and MITIGATION_BEFORE is not None
            else "Mitigation results are unavailable."
        )
    )

    qwen_answer = "The local Qwen comparison did not complete."
    if not MODEL_COMPARISON.empty and len(MODEL_COMPARISON) >= 2:
        g_acc = float(MODEL_COMPARISON.loc[MODEL_COMPARISON["model"]=="Gemini Flash","accuracy"].iloc[0])
        q_row = MODEL_COMPARISON[MODEL_COMPARISON["model"]!="Gemini Flash"].iloc[0]
        q_acc = float(q_row["accuracy"])
        q_lat = float(q_row["mean_latency_s"])
        g_lat = float(MODEL_COMPARISON.loc[MODEL_COMPARISON["model"]=="Gemini Flash","mean_latency_s"].iloc[0])

        if q_acc >= g_acc - 0.05:
            qwen_answer = (
                f"Qwen is competitive for the narrow classification role: accuracy {q_acc:.3f} "
                f"versus Gemini {g_acc:.3f}. Mean per-item latency was {q_lat:.3f}s locally "
                f"versus {g_lat:.3f}s for the batched Gemini baseline. This supports keeping Qwen "
                f"as an optional local classifier, not as the full brief generator."
            )
        else:
            qwen_answer = (
                f"Qwen was not accurate enough to replace Gemini even for the narrow classification role: "
                f"accuracy {q_acc:.3f} versus Gemini {g_acc:.3f}. Its measured mean latency was {q_lat:.3f}s "
                f"per item. The final architecture should keep the local model out of the critical extraction path."
            )

    judge_text = "\n".join(
        f"- **{idx}:** {float(row['average']):.3f}"
        for idx, row in JUDGE_SUMMARY.iterrows()
    )

    selected_reason = (
        f"Configuration **{SELECTED_CONFIG}** had the highest deterministic aggregate extraction score "
        f"({float(SUMMARY.loc[SELECTED_CONFIG,'extraction_score']):.3f}) and was therefore selected "
        f"for the 10-case judge pass."
    )

    note = f"""
# Research Note — Meeting-to-Action Assistant

## Research question

Can a compact meeting-to-action pipeline improve extraction of operational information by adding retrieved/reranked guidance and dynamic few-shot examples, and can a small local Qwen model replace Gemini for any narrow part of the workflow?

## Measured results

### A/B/C deterministic metrics

{det.to_markdown()}

### LLM-as-judge

The selected configuration was **{SELECTED_CONFIG}**. Judge averages over the 10 held-out cases:

{judge_text}

### Questions and answers

**Which configuration performed best?**  
{selected_reason}

**Did retrieval help?**  
{retrieval_answer}

**Did dynamic few-shot prompting help?**  
{fewshot_answer}

**What failure was identified, and which component caused it?**  
The traced failure case was **{FAILURE['case_id']}**. The ablation evidence attributed it to **{FAILURE['component']}**.

**Did the mitigation help?**  
{mitigation_answer}

**How did the small local Qwen model compare with Gemini?**  
{qwen_answer}

**What should the final architecture be?**  
Keep the simplest measured configuration that produced the best held-out extraction score. Use Gemini for the full meeting brief because that stage requires ambiguous decision interpretation, owner/coreference reasoning, and synthesis. Keep embeddings, FAISS retrieval, cross-encoder reranking, validation, matching, and metrics deterministic/local. Use Qwen only if the measured narrow classification accuracy is close enough to Gemini to justify the local role.

## Failure mitigation

The mitigation is deliberately narrow: predicted decisions whose evidence contains suggestion language without a commitment cue are filtered. This targets the documented suggestion-as-decision failure without replacing the model's broader reasoning.

## Conclusion

The final decision is based on the measurements above rather than assuming that more retrieval, more examples, or a local model must improve the system.
"""
    display(Markdown(note))
    return note

RESEARCH_NOTE = build_research_note()

## Submission guard and cache export

The final notebook should be saved with outputs and exported to HTML only after the guard reports `READY FOR SUBMISSION`.

The cache export can be pasted back into `PRELOADED_CACHE_JSON` before a later clean rerun so completed Gemini calls do not have to be repeated.

In [ ]:
problems = []

if not RESULTS:
    problems.append("A/B/C evaluation missing")

if "METRICS" not in globals() or len(METRICS) != len(EVAL_CASES) * 3:
    problems.append("A/B/C deterministic metrics incomplete")

if JUDGE_DF.empty or len(JUDGE_DF) != len(EVAL_CASES):
    problems.append("10-case LLM-as-judge pass incomplete")

if FAILURE is None:
    problems.append("failure trace missing")

if RESEARCH_NOTE is None:
    problems.append("dynamic research note missing")

if MODEL_COMPARISON.empty or len(MODEL_COMPARISON) < 2:
    problems.append("Gemini vs local Qwen comparison incomplete")

if problems:
    print("NOT READY FOR SUBMISSION")
    for p in problems:
        print("-", p)
else:
    print("READY FOR SUBMISSION")
    print("- 10 held-out cases × 3 A/B/C configurations")
    print("- concrete deterministic metrics reported")
    print("- 10-case LLM-as-judge pass complete")
    print("- failure traced to a component with ablation/log evidence")
    print("- mitigation measured")
    print("- Gemini vs local Qwen comparison measured")
    print("- dynamic research note generated from measured results")

CACHE_EXPORT = json.dumps(CACHE, separators=(",",":"), ensure_ascii=False)
print("\nCACHE_EXPORT =", CACHE_EXPORT)
print("\nHTML export:")
print("jupyter nbconvert --to html week8_meeting_to_action_capstone.ipynb")